# Learning Urban Crimes Representation

## Aprendizaje de representaciones urbanas
Input:  firmas_h3.csv (1,061 hexágonos × 45 dims)
        h3_metadata.csv (metadatos por hexágono)

Métodos:
  A) Línea base — PCA, NMF, K-Means, GMM, HDBSCAN
  B) Autoencoder denso
  C) Comparación de representaciones

Output: embeddings_h3.csv — embeddings finales por hexágono
        clusters_h3.csv — asignación de clusters por método
        modelo_autoencoder.pth — pesos del autoencoder

### Paquetes

In [28]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA, NMF
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, adjusted_rand_score, normalized_mutual_info_score
import hdbscan

# autoencodes
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import warnings
warnings.filterwarnings('ignore')

### Ejecución

#### Cargar datos

In [29]:
print("Cargando firmas...")
firmas = pd.read_csv("../data/auxiliar/firmas_h3.csv", index_col='h3_id')
metadata = pd.read_csv("../data/auxiliar/h3_metadata.csv", index_col='h3_id')

print(f"Hexágonos: {len(firmas):,}")
print(f"Dimensiones: {firmas.shape[1]}")

Cargando firmas...
Hexágonos: 1,061
Dimensiones: 45


#### Implementación

In [30]:
# ============================================================================
# PASO 1: Preprocesamiento
# ============================================================================
# Separar features proporcionales de features absolutas
cols_proporciones = [c for c in firmas.columns if c not in ['intensidad_log', 'ratio_violencia']]
cols_extra = ['intensidad_log', 'ratio_violencia']

# Para PCA y Autoencoder: estandarizar todo
scaler_std = StandardScaler()
X_std = scaler_std.fit_transform(firmas)

# Para NMF: usar MinMax (NMF requiere valores no negativos)
scaler_mm = MinMaxScaler()
X_mm = scaler_mm.fit_transform(firmas)

print(f"Datos estandarizados: {X_std.shape}")

Datos estandarizados: (1061, 45)


In [31]:
# ============================================================================
# PASO 2A: PCA
# ============================================================================
print(f"\n{'='*80}")
print(f"Método A1: PCA")
print(f"{'='*80}")

# PCA completo para ver varianza explicada
pca_full = PCA().fit(X_std)
varianza_acum = np.cumsum(pca_full.explained_variance_ratio_)

# ¿Cuántas componentes para 80%, 90%, 95%?
for umbral in [0.80, 0.90, 0.95]:
    n_comp = np.argmax(varianza_acum >= umbral) + 1
    print(f"  Componentes para {umbral*100:.0f}% varianza: {n_comp}")

# Usar componentes que expliquen 90% de la varianza
N_COMPONENTS_PCA = np.argmax(varianza_acum >= 0.90) + 1
pca = PCA(n_components=N_COMPONENTS_PCA)
X_pca = pca.fit_transform(X_std)

print(f"\n  PCA con {N_COMPONENTS_PCA} componentes:")
print(f"  Varianza explicada: {pca.explained_variance_ratio_.sum()*100:.1f}%")

# Top features por componente (primeras 3)
feature_names = firmas.columns.tolist()
print(f"\n  Top features por componente:")
for i in range(min(3, N_COMPONENTS_PCA)):
    loadings = pd.Series(pca.components_[i], index=feature_names)
    top_pos = loadings.nlargest(3)
    top_neg = loadings.nsmallest(3)
    print(f"\n  PC{i+1} ({pca.explained_variance_ratio_[i]*100:.1f}% varianza):")
    print(f"    (+) {', '.join([f'{n}: {v:.3f}' for n, v in top_pos.items()])}")
    print(f"    (-) {', '.join([f'{n}: {v:.3f}' for n, v in top_neg.items()])}")


Método A1: PCA
  Componentes para 80% varianza: 22
  Componentes para 90% varianza: 29
  Componentes para 95% varianza: 34

  PCA con 29 componentes:
  Varianza explicada: 90.7%

  Top features por componente:

  PC1 (16.5% varianza):
    (+) delito_violencia_familiar: 0.309, dia_sunday: 0.264, hora_noche: 0.246
    (-) delito_falsificacion_y_documentos: -0.284, delito_robo_sin_violencia: -0.281, delito_fraude_y_delitos_patrimoniales: -0.217

  PC2 (8.1% varianza):
    (+) delito_robo_con_violencia: 0.423, ratio_violencia: 0.385, intensidad_log: 0.248
    (-) hora_manana: -0.346, delito_fraude_y_delitos_patrimoniales: -0.289, delito_delitos_ambientales: -0.244

  PC3 (5.3% varianza):
    (+) delito_lesiones_culposas: 0.345, delito_homicidio: 0.310, delito_dano_en_propiedad: 0.256
    (-) delito_delitos_de_servidores_publicos: -0.276, dia_monday: -0.225, delito_amenazas: -0.214


In [32]:
# ============================================================================
# PASO 2B: NMF
# ============================================================================
print(f"\n{'='*80}")
print(f"Método A2: Non-Negative Matrix Factorization (NMF)")
print(f"{'='*80}")

# NMF con el mismo número de componentes
nmf = NMF(n_components=N_COMPONENTS_PCA, init='nndsvda', max_iter=500, random_state=42)
X_nmf = nmf.fit_transform(X_mm)
print(f"  NMF con {N_COMPONENTS_PCA} componentes")
print(f"  Error de reconstrucción: {nmf.reconstruction_err_:.2f}")

# Top features por tema NMF
print(f"\n  Temas NMF (top features por componente):")
for i in range(min(5, N_COMPONENTS_PCA)):
    loadings = pd.Series(nmf.components_[i], index=feature_names)
    top = loadings.nlargest(5)
    print(f"\n  Tema {i+1}:")
    for n, v in top.items():
        print(f"    {n:45s} {v:.3f}")


Método A2: Non-Negative Matrix Factorization (NMF)
  NMF con 29 componentes
  Error de reconstrucción: 7.34

  Temas NMF (top features por componente):

  Tema 1:
    hora_tarde                                    8.095
    trim_Q4                                       7.360
    dia_saturday                                  3.824
    dia_wednesday                                 2.654
    delito_fraude_y_delitos_patrimoniales         1.866

  Tema 2:
    delito_falsificacion_y_documentos             6.217
    trim_Q4                                       0.903
    dia_friday                                    0.875
    intensidad_log                                0.796
    dia_wednesday                                 0.631

  Tema 3:
    delito_robo_con_violencia                     7.226
    ratio_violencia                               5.879
    dia_tuesday                                   1.787
    dia_friday                                    1.649
    hora_noche                

In [33]:
# ============================================================================
# PASO 2C: Clustering — K-Means
# ============================================================================
print(f"\n{'='*80}")
print(f"Método A3: K-Means (sobre embeddings PCA)")
print(f"{'='*80}")

# Buscar K óptimo
resultados_k = []
for k in range(3, 16):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_pca)
    sil = silhouette_score(X_pca, labels)
    ch = calinski_harabasz_score(X_pca, labels)
    inertia = km.inertia_
    resultados_k.append({'k': k, 'silhouette': sil, 'calinski_harabasz': ch, 'inertia': inertia})
    print(f"  K={k:2d}  Silhouette={sil:.3f}  Calinski-Harabasz={ch:.1f}")

# Elegir mejor K por silhouette
df_k = pd.DataFrame(resultados_k)
mejor_k = df_k.loc[df_k['silhouette'].idxmax(), 'k']
print(f"\n  Mejor K por Silhouette: {int(mejor_k)}")

# Entrenar K-Means final
km_final = KMeans(n_clusters=int(mejor_k), n_init=20, random_state=42)
clusters_km = km_final.fit_predict(X_pca)
print(f"  Distribución de clusters K-Means:")
for c, count in pd.Series(clusters_km).value_counts().sort_index().items():
    print(f"    Cluster {c}: {count} hexágonos ({count/len(clusters_km)*100:.1f}%)")


Método A3: K-Means (sobre embeddings PCA)
  K= 3  Silhouette=0.107  Calinski-Harabasz=120.2
  K= 4  Silhouette=0.112  Calinski-Harabasz=96.4
  K= 5  Silhouette=0.060  Calinski-Harabasz=84.3
  K= 6  Silhouette=0.072  Calinski-Harabasz=76.8
  K= 7  Silhouette=0.072  Calinski-Harabasz=68.5
  K= 8  Silhouette=0.049  Calinski-Harabasz=63.4
  K= 9  Silhouette=0.061  Calinski-Harabasz=59.8
  K=10  Silhouette=0.040  Calinski-Harabasz=54.8
  K=11  Silhouette=0.066  Calinski-Harabasz=52.3
  K=12  Silhouette=0.061  Calinski-Harabasz=50.0
  K=13  Silhouette=0.038  Calinski-Harabasz=47.1
  K=14  Silhouette=0.026  Calinski-Harabasz=45.9
  K=15  Silhouette=0.035  Calinski-Harabasz=43.8

  Mejor K por Silhouette: 4
  Distribución de clusters K-Means:
    Cluster 0: 302 hexágonos (28.5%)
    Cluster 1: 270 hexágonos (25.4%)
    Cluster 2: 41 hexágonos (3.9%)
    Cluster 3: 448 hexágonos (42.2%)


In [34]:
# ============================================================================
# PASO 2D: GMM
# ============================================================================
print(f"\n{'='*80}")
print(f"Método A4: Gaussian Mixture Model (GMM)")
print(f"{'='*80}")

resultados_gmm = []
for k in range(3, 16):
    gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=42, n_init=3)
    labels = gmm.fit_predict(X_pca)
    sil = silhouette_score(X_pca, labels)
    bic = gmm.bic(X_pca)
    aic = gmm.aic(X_pca)
    resultados_gmm.append({'k': k, 'silhouette': sil, 'bic': bic, 'aic': aic})
    print(f"  K={k:2d}  Silhouette={sil:.3f}  BIC={bic:.0f}  AIC={aic:.0f}")

df_gmm = pd.DataFrame(resultados_gmm)
mejor_k_gmm = df_gmm.loc[df_gmm['bic'].idxmin(), 'k']
print(f"\n  Mejor K por BIC: {int(mejor_k_gmm)}")

gmm_final = GaussianMixture(n_components=int(mejor_k_gmm), covariance_type='full',
                             random_state=42, n_init=5)
clusters_gmm = gmm_final.fit_predict(X_pca)
probs_gmm = gmm_final.predict_proba(X_pca)


Método A4: Gaussian Mixture Model (GMM)
  K= 3  Silhouette=0.145  BIC=80405  AIC=73481
  K= 4  Silhouette=0.110  BIC=79404  AIC=70171
  K= 5  Silhouette=0.086  BIC=82901  AIC=71358
  K= 6  Silhouette=0.060  BIC=83067  AIC=69214
  K= 7  Silhouette=0.024  BIC=84606  AIC=68443
  K= 8  Silhouette=0.064  BIC=81637  AIC=63165
  K= 9  Silhouette=0.078  BIC=87492  AIC=66710
  K=10  Silhouette=0.033  BIC=88628  AIC=65537
  K=11  Silhouette=0.069  BIC=90246  AIC=64845
  K=12  Silhouette=0.022  BIC=90634  AIC=62923
  K=13  Silhouette=0.055  BIC=94333  AIC=64313
  K=14  Silhouette=0.051  BIC=94064  AIC=61734
  K=15  Silhouette=0.017  BIC=95156  AIC=60516

  Mejor K por BIC: 4


In [35]:
# ============================================================================
# PASO 2E: HDBSCAN
# ============================================================================
print(f"\n{'='*80}")
print(f"Método A5: HDBSCAN")
print(f"{'='*80}")

# Análisis de sensibilidad: encontrar parámetros óptimos
best_result = None
best_n_clusters = 0

for min_size in [5, 10, 15, 20]:
    for min_samples in [2, 3, 5]:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=min_size,
            min_samples=min_samples,
            metric='euclidean',
            cluster_selection_method='leaf'  # 'leaf' es más permisivo que 'eom'
        )
        labels = clusterer.fit_predict(X_pca)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = (labels == -1).sum()
        
        # Preferir configuraciones que encuentren clusters (no ruido puro)
        if n_clusters > best_n_clusters:
            best_n_clusters = n_clusters
            best_result = {
                'min_size': min_size,
                'min_samples': min_samples,
                'labels': labels,
                'n_clusters': n_clusters,
                'n_noise': n_noise
            }
        
        if n_clusters > 0:
            print(f"    min_size={min_size}, min_samples={min_samples}: "
                    f"{n_clusters} clusters, {n_noise} ruido ({n_noise/len(labels)*100:.1f}%)")

if best_result is None:
    print(f"No se encontraron clusters. Reducción de cluster_size y min_samples.")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=5,
        min_samples=1,
        metric='euclidean',
        cluster_selection_method='leaf'
    )
    clusters_hdb = clusterer.fit_predict(X_pca)
else:
    print(f"\n  Mejor configuración: min_size={best_result['min_size']}, "
            f"min_samples={best_result['min_samples']}")
    clusters_hdb = best_result['labels']

n_clusters_hdb = len(set(clusters_hdb)) - (1 if -1 in clusters_hdb else 0)
n_noise = (clusters_hdb == -1).sum()

print(f"\n  Clusters encontrados: {n_clusters_hdb}")
print(f"  Puntos de ruido: {n_noise} ({n_noise/len(clusters_hdb)*100:.1f}%)")

if n_clusters_hdb > 1:
    mask_valid = clusters_hdb != -1
    if mask_valid.sum() > 0:
        sil_hdb = silhouette_score(X_pca[mask_valid], clusters_hdb[mask_valid])
        print(f"  Silhouette (sin ruido): {sil_hdb:.3f}")

print(f"  Distribución:")
for c, count in pd.Series(clusters_hdb).value_counts().sort_index().items():
    label = f"Cluster {c}" if c >= 0 else "Ruido"
    print(f"    {label}: {count} hexágonos ({count/len(clusters_hdb)*100:.1f}%)")



Método A5: HDBSCAN
    min_size=5, min_samples=2: 4 clusters, 948 ruido (89.3%)
    min_size=5, min_samples=3: 5 clusters, 1025 ruido (96.6%)
    min_size=10, min_samples=2: 3 clusters, 955 ruido (90.0%)

  Mejor configuración: min_size=5, min_samples=3

  Clusters encontrados: 5
  Puntos de ruido: 1025 (96.6%)
  Silhouette (sin ruido): 0.336
  Distribución:
    Ruido: 1025 hexágonos (96.6%)
    Cluster 0: 7 hexágonos (0.7%)
    Cluster 1: 7 hexágonos (0.7%)
    Cluster 2: 9 hexágonos (0.8%)
    Cluster 3: 7 hexágonos (0.7%)
    Cluster 4: 6 hexágonos (0.6%)


In [36]:

# ============================================================================
# PASO 3: AUTOENCODER DENSO
# ============================================================================
print(f"\n{'='*80}")
print(f"Método B: Autoencoder denso")
print(f"{'='*80}")

# Arquitectura
INPUT_DIM = X_std.shape[1]  # 45
ENCODING_DIM = 12  # embedding de salida
HIDDEN_DIMS = [32, 24]  # capas intermedias

class CrimeAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dims, encoding_dim):
        super().__init__()
        
        # Encoder
        encoder_layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            encoder_layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.ReLU(),
                nn.Dropout(0.1)
            ])
            prev_dim = h_dim
        encoder_layers.append(nn.Linear(prev_dim, encoding_dim))
        self.encoder = nn.Sequential(*encoder_layers)
        
        # Decoder (simétrico)
        decoder_layers = []
        prev_dim = encoding_dim
        for h_dim in reversed(hidden_dims):
            decoder_layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.ReLU(),
                nn.Dropout(0.1)
            ])
            prev_dim = h_dim
        decoder_layers.append(nn.Linear(prev_dim, input_dim))
        self.decoder = nn.Sequential(*decoder_layers)
    
    def encode(self, x):
        return self.encoder(x)
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        z = self.encode(x)
        x_recon = self.decode(z)
        return x_recon, z

# Preparar datos
X_tensor = torch.FloatTensor(X_std)
dataset = TensorDataset(X_tensor, X_tensor)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Entrenar
model = CrimeAutoencoder(INPUT_DIM, HIDDEN_DIMS, ENCODING_DIM)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5)
criterion = nn.MSELoss()

EPOCHS = 300
best_loss = float('inf')
patience_counter = 0
PATIENCE = 50

print(f"  Arquitectura: {INPUT_DIM} → {HIDDEN_DIMS} → {ENCODING_DIM} → {list(reversed(HIDDEN_DIMS))} → {INPUT_DIM}")
print(f"  Entrenando ({EPOCHS} epochs max, early stopping patience={PATIENCE})...")

losses = []
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for batch_x, _ in dataloader:
        optimizer.zero_grad()
        x_recon, z = model(batch_x)
        loss = criterion(x_recon, batch_x)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    scheduler.step(avg_loss)
    
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        best_state = model.state_dict().copy()
    else:
        patience_counter += 1
    
    if patience_counter >= PATIENCE:
        print(f"  Early stopping en epoch {epoch+1}")
        break
    
    if (epoch + 1) % 50 == 0:
        print(f"    Epoch {epoch+1:3d}: loss = {avg_loss:.6f}")

# Cargar mejor modelo
model.load_state_dict(best_state)
print(f"  Mejor loss: {best_loss:.6f}")

# Extraer embeddings
model.eval()
with torch.no_grad():
    x_recon, embeddings_ae = model(X_tensor)
    recon_error = torch.mean((X_tensor - x_recon) ** 2, dim=1).numpy()

X_ae = embeddings_ae.numpy()
print(f"  Embeddings shape: {X_ae.shape}")
print(f"  Error de reconstrucción medio: {recon_error.mean():.4f}")
print(f"  Error de reconstrucción máximo: {recon_error.max():.4f}")

# Guardar modelo
torch.save(best_state, '../models/modelo_autoencoder.pth')
print(f"  Modelo guardado: ../models/modelo_autoencoder.pth")



Método B: Autoencoder denso
  Arquitectura: 45 → [32, 24] → 12 → [24, 32] → 45
  Entrenando (300 epochs max, early stopping patience=50)...
    Epoch  50: loss = 0.634293
    Epoch 100: loss = 0.589322
    Epoch 150: loss = 0.578496
    Epoch 200: loss = 0.563093
    Epoch 250: loss = 0.560571
  Early stopping en epoch 283
  Mejor loss: 0.552450
  Embeddings shape: (1061, 12)
  Error de reconstrucción medio: 0.4549
  Error de reconstrucción máximo: 5.4512
  Modelo guardado: ../models/modelo_autoencoder.pth


In [37]:

# ============================================================================
# PASO 4: Clustering sobre embeddings del autoencoder
# ============================================================================
print(f"\n{'='*80}")
print(f"Clustering sobre embeddings del Autoencoder")
print(f"{'='*80}")

# K-Means sobre embeddings AE
resultados_ae_k = []
for k in range(3, 16):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_ae)
    sil = silhouette_score(X_ae, labels)
    resultados_ae_k.append({'k': k, 'silhouette': sil})
    print(f"  K={k:2d}  Silhouette={sil:.3f}")

df_ae_k = pd.DataFrame(resultados_ae_k)
mejor_k_ae = df_ae_k.loc[df_ae_k['silhouette'].idxmax(), 'k']
print(f"\n  Mejor K por Silhouette (AE): {int(mejor_k_ae)}")

km_ae = KMeans(n_clusters=int(mejor_k_ae), n_init=20, random_state=42)
clusters_ae = km_ae.fit_predict(X_ae)


Clustering sobre embeddings del Autoencoder
  K= 3  Silhouette=0.161
  K= 4  Silhouette=0.166
  K= 5  Silhouette=0.175
  K= 6  Silhouette=0.174
  K= 7  Silhouette=0.154
  K= 8  Silhouette=0.158
  K= 9  Silhouette=0.162
  K=10  Silhouette=0.176
  K=11  Silhouette=0.158
  K=12  Silhouette=0.153
  K=13  Silhouette=0.162
  K=14  Silhouette=0.132
  K=15  Silhouette=0.131

  Mejor K por Silhouette (AE): 10


In [38]:
# ============================================================================
# PASO 5: Comparación de métodos
# ============================================================================
print(f"\n{'='*80}")
print(f"Comparación de representaciones y clusterings")
print(f"{'='*80}")

# Silhouette de cada representación con su mejor clustering
print(f"\n  {'Método':<25s} {'Dims':>5s} {'Clusters':>9s} {'Silhouette':>11s}")
print(f"  {'-'*55}")

metodos_repr = {
    'PCA': (X_pca, clusters_km),
    'NMF': (X_nmf, KMeans(n_clusters=int(mejor_k), n_init=10, random_state=42).fit_predict(X_nmf)),
    'Autoencoder': (X_ae, clusters_ae),
}

for nombre, (X_repr, labels) in metodos_repr.items():
    n_cl = len(set(labels)) - (1 if -1 in labels else 0)
    sil = silhouette_score(X_repr, labels) if n_cl > 1 else 0
    print(f"  {nombre:<25s} {X_repr.shape[1]:>5d} {n_cl:>9d} {sil:>11.3f}")

# Concordancia entre clusterings
print(f"\n  Concordancia entre métodos (Adjusted Rand Index):")
nombres = ['KMeans-PCA', 'GMM-PCA', 'KMeans-AE']
todos_clusters = [clusters_km, clusters_gmm, clusters_ae]
if not all(c == -1 for c in clusters_hdb):
    nombres.append('HDBSCAN')
    todos_clusters.append(clusters_hdb)

for i in range(len(nombres)):
    for j in range(i+1, len(nombres)):
        ari = adjusted_rand_score(todos_clusters[i], todos_clusters[j])
        nmi = normalized_mutual_info_score(todos_clusters[i], todos_clusters[j])
        print(f"  {nombres[i]:15s} vs {nombres[j]:15s}  ARI={ari:.3f}  NMI={nmi:.3f}")


Comparación de representaciones y clusterings

  Método                     Dims  Clusters  Silhouette
  -------------------------------------------------------
  PCA                          29         4       0.112
  NMF                          29         4       0.173
  Autoencoder                  12        10       0.176

  Concordancia entre métodos (Adjusted Rand Index):
  KMeans-PCA      vs GMM-PCA          ARI=0.380  NMI=0.367
  KMeans-PCA      vs KMeans-AE        ARI=0.311  NMI=0.426
  KMeans-PCA      vs HDBSCAN          ARI=0.000  NMI=0.048
  GMM-PCA         vs KMeans-AE        ARI=0.207  NMI=0.311
  GMM-PCA         vs HDBSCAN          ARI=0.003  NMI=0.051
  KMeans-AE       vs HDBSCAN          ARI=-0.002  NMI=0.056


In [39]:
# ============================================================================
# PASO 6: Perfiles de clusters (usando Autoencoder como representación principal)
# ============================================================================
print(f"\n{'='*80}")
print(f"Perfiles de clusters (Autoencoder + K-Means)")
print(f"{'='*80}")

firmas['cluster_ae'] = clusters_ae
firmas_con_meta = firmas.join(metadata)

for c in sorted(firmas['cluster_ae'].unique()):
    grupo = firmas_con_meta[firmas_con_meta['cluster_ae'] == c]
    n = len(grupo)
    print(f"\n- Cluster {c} — {n} hexágonos ({n/len(firmas)*100:.1f}%)")
    
    # Alcaldías dominantes
    alcs = grupo['alcaldia_dominante'].value_counts().head(3)
    print(f"  Alcaldías: {', '.join([f'{a} ({c_})' for a, c_ in alcs.items()])}")
    
    # Top delitos
    delito_cols = [col for col in firmas.columns if col.startswith('delito_')]
    top_delitos = grupo[delito_cols].mean().sort_values(ascending=False).head(5)
    print(f"  Perfil delictivo:")
    for col, val in top_delitos.items():
        nombre = col.replace('delito_', '').replace('_', ' ').upper()
        print(f"    {nombre:40s} {val*100:5.1f}%")
    
    # Intensidad y violencia
    print(f"  Intensidad media: {grupo['intensidad_log'].mean():.2f} (~{np.expm1(grupo['intensidad_log'].mean()):.0f} registros)")
    print(f"  Ratio violencia: {grupo['ratio_violencia'].mean():.3f}")

# Limpiar columna temporal
firmas.drop('cluster_ae', axis=1, inplace=True)


Perfiles de clusters (Autoencoder + K-Means)

- Cluster 0 — 406 hexágonos (38.3%)
  Alcaldías: GUSTAVO A. MADERO (60), TLALPAN (46), COYOACAN (42)
  Perfil delictivo:
    ROBO SIN VIOLENCIA                        27.7%
    VIOLENCIA FAMILIAR                        14.5%
    FRAUDE Y DELITOS PATRIMONIALES            13.7%
    ROBO CON VIOLENCIA                        11.3%
    AMENAZAS                                   7.4%
  Intensidad media: 7.55 (~1894 registros)
  Ratio violencia: 0.143

- Cluster 1 — 252 hexágonos (23.8%)
  Alcaldías: TLALPAN (53), XOCHIMILCO (47), MILPA ALTA (37)
  Perfil delictivo:
    VIOLENCIA FAMILIAR                        25.6%
    ROBO SIN VIOLENCIA                        16.1%
    FRAUDE Y DELITOS PATRIMONIALES            11.8%
    AMENAZAS                                   9.5%
    ROBO CON VIOLENCIA                         8.5%
  Intensidad media: 6.14 (~461 registros)
  Ratio violencia: 0.135

- Cluster 2 — 7 hexágonos (0.7%)
  Alcaldías: XOCHIMILCO (2

In [40]:
# ============================================================================
# PASO 7: Exportar embeddings y clusters
# ============================================================================

# Embeddings (autoencoder como principal, PCA como línea base)
embeddings_df = pd.DataFrame(
    X_ae,
    index=firmas.index,
    columns=[f'emb_ae_{i}' for i in range(X_ae.shape[1])]
)
# Añadir PCA también
for i in range(X_pca.shape[1]):
    embeddings_df[f'emb_pca_{i}'] = X_pca[:, i]

# Añadir error de reconstrucción (útil para la detección de disparidad final)
embeddings_df['recon_error'] = recon_error

embeddings_df.index.name = 'h3_id'
embeddings_df.to_csv('../data/results/embeddings_h3.csv', encoding='utf-8-sig')
print(f"../data/results/embeddings_h3.csv ({len(embeddings_df)} hexágonos × {embeddings_df.shape[1]} columnas)")

# Clusters
clusters_df = pd.DataFrame({
    'cluster_kmeans_pca': clusters_km,
    'cluster_gmm_pca': clusters_gmm,
    'cluster_hdbscan': clusters_hdb,
    'cluster_kmeans_ae': clusters_ae,
}, index=firmas.index)
clusters_df.index.name = 'h3_id'
clusters_df.to_csv('../data/auxiliar/clusters_h3.csv', encoding='utf-8-sig')
print(f"../data/auxiliar/clusters_h3.csv ({len(clusters_df)} hexágonos × {clusters_df.shape[1]} métodos)")

# Métricas de entrenamiento
pd.DataFrame({'epoch': range(1, len(losses)+1), 'loss': losses}).to_csv(
    '../data/results/autoencoder_training_loss.csv', index=False
)
print(f"../data/results/autoencoder_training_loss.csv ({len(losses)} epochs)")

../data/results/embeddings_h3.csv (1061 hexágonos × 42 columnas)
../data/auxiliar/clusters_h3.csv (1061 hexágonos × 4 métodos)
../data/results/autoencoder_training_loss.csv (283 epochs)
